# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook provides a step-by-step workflow for loading and exploring the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.y7m0-f273) using the [`mlcroissant`](https://github.com/mlcommons/croissant-python) library.

### Dataset Source
The dataset source is a Croissant schema located at the following URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed (uncomment the next line if running in a new environment)
!pip install mlcroissant

## 1. Data Loading
Load the dataset schema and metadata using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Initialize the Dataset object
dataset = mlc.Dataset(croissant_url)

# Print human-readable metadata summary
meta = dataset.metadata
print(f"{meta.name}: {meta.description}\n\nIdentifier: {meta.identifier}\nLicense: {meta.license}\nVersion: {meta.version}\n")


## 2. Data Overview
Explore which record sets, fields, and columns are available in the dataset, referencing all entities by their `@id`.

In [ ]:
# Show all available record sets by @id
record_set_ids = []
for recordset in dataset.record_sets:
    print(f"RecordSet name: {recordset.name} | @id: {recordset.id}")
    record_set_ids.append(recordset.id)
    print("  Fields:")
    for field in recordset.fields:
        print(f"    Field name: {field.name} | @id: {field.id} | Data type: {field.data_type}")
    print()

if not record_set_ids:
    print("WARNING: No record sets were found in the Croissant metadata. This Croissant package may reference record sets in distribution/data files which should be inspected in the next steps.")

## 3. Data Extraction
The next step is to attempt to extract data by loading records from each record set using its `@id`.

**Note:** If the previous cell listed no record sets, you may need to review dataset distributions or files instead.

In [ ]:
# List to hold DataFrames for each record set
dataframes = {}

# Loop through all record sets by @id and load into a DataFrame
if len(record_set_ids) > 0:
    for rsid in record_set_ids:
        # Records is a generator of dicts keyed by field @id
        records = list(dataset.records(record_set=rsid))
        df = pd.DataFrame(records)
        dataframes[rsid] = df
        print(f'RecordSet @id: {rsid}')
        print(f'Fields (column @ids): {list(df.columns)}')
        display(df.head(3))
    # Pick the first record set for further demo
    main_record_set_id = record_set_ids[0]
else:
    print("No record sets found. Check the dataset's distributions directly or examine the raw files with mlcroissant's lower-level tools.")
    main_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Conduct typical data wrangling and analysis steps, referencing fields and columns by their `@id`.

In [ ]:
import numpy as np
pd.set_option('mode.chained_assignment', None)  # To suppress SettingWithCopyWarning during demo

if main_record_set_id and len(dataframes) > 0:
    df = dataframes[main_record_set_id]
    # Attempt to infer a numeric field @id (e.g., log-likelihood or coefficient)
    # For illustration: pick the first numeric column found
    numeric_field_id = None
    for col in df.columns:
        # Try to infer numeric fields by their dtype (if loaded)
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    if numeric_field_id:
        threshold = df[numeric_field_id].mean() if np.isfinite(df[numeric_field_id].mean()) else 0
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records where {numeric_field_id} > {threshold:.2f} (using field @id).\n")
        print(filtered_df.head())
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        # Try to group by a plausible categorical/id field
        group_field_id = None
        for col in df.columns:
            unique_vals = df[col].nunique(dropna=True)
            if unique_vals < 20 and col != numeric_field_id:
                group_field_id = col
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped data by {group_field_id} (using field @id):")
            print(grouped_df.head())
        else:
            print("No suitable categorical/grouping field detected by heuristics.")
    else:
        print("No numeric field inferred in the first record set. Please visually inspect DataFrame columns or check the raw data.")
else:
    print("No data loaded for EDA step.")

## 5. Visualization
Visualize distributions or relationships between fields using matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and len(dataframes) > 0 and numeric_field_id:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f'Distribution of numeric field ({numeric_field_id})')
    plt.xlabel(numeric_field_id)
    plt.show()

    # If group_field_id detected, show grouped boxplot
    if 'group_field_id' in locals() and group_field_id:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f'{numeric_field_id} by {group_field_id} (@id)')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No numeric or groupable fields detected for visualization.")

## 6. Conclusion
In this notebook, we loaded and explored the FAIR^2 dataset using its Croissant schema with the `mlcroissant` library. We discovered available record sets and fields (referenced by `@id`), loaded records into DataFrames, filtered and normalized a numeric column, grouped by categorical fields where possible, and visualized the primary variable distributions. For richer exploration or publication, see the data dictionary or Croissant schema directly for further `@id` reference and definitions.